[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somanshusingla/llm-from-scratch/blob/main/notebooks/03_ch4_gpt_model.ipynb)

# Chapter 4 — Implementing a GPT Model from Scratch

**Build a Large Language Model (From Scratch)** · notebook 3 of 7

Time to assemble a **real GPT-2 (124M parameter) architecture**. You already have
the hardest piece — multi-head attention. Here we add the remaining building
blocks and stack them:

- **Layer normalization** — keeps activations well-behaved.
- **GELU + feed-forward network** — the per-token "thinking" layer.
- **Shortcut (residual) connections** — let gradients flow through deep stacks.
- **Transformer block** — attention + feed-forward, wired with norms + shortcuts.
- **GPTModel** — embeddings → N transformer blocks → output head.
- **Text generation** — turn model outputs into new tokens.

At the end you'll instantiate a 124-million-parameter model and generate text.
It'll be gibberish — the weights are random! Training comes in Chapter 5.

> This notebook is **self-contained**: it re-defines the attention class from
> Chapter 3 so you can run it on its own (e.g. on Colab).

*Building + a forward pass of the 124M model runs on CPU in a few seconds.*

In [1]:
import importlib, subprocess, sys
try:
    import tiktoken
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken"], check=True)
    import tiktoken

import torch
import torch.nn as nn
tokenizer = tiktoken.get_encoding("gpt2")
print("torch:", torch.__version__, "| tiktoken:", tiktoken.__version__)

torch: 2.11.0+cu128 | tiktoken: 0.13.0


## The configuration

Every dimension of the model comes from this one dictionary. These are the exact
GPT-2 "small" (124M) settings.

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # BPE vocabulary size (from Chapter 2)
    "context_length": 1024, # max number of tokens the model can attend over
    "emb_dim": 768,         # embedding / hidden dimension
    "n_heads": 12,          # number of attention heads
    "n_layers": 12,         # number of transformer blocks
    "drop_rate": 0.1,       # dropout rate
    "qkv_bias": False,      # bias in Q/K/V projections (off for now)
}

## Recap: Multi-Head Attention (from Chapter 3)

Same class you wrote in the previous notebook — included here so this notebook
stands alone.

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys    = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim)
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim)
        values  = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim)
        keys, queries, values = keys.transpose(1, 2), queries.transpose(1, 2), values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

## 4.2 Normalizing activations with layer normalization

Layer norm rescales each token's vector to mean 0 and variance 1, then applies a
learnable `scale` and `shift`. This stabilizes and speeds up training. (We use
`unbiased=False` for the variance to match GPT-2's behavior.)

In [4]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


# quick check: output has ~0 mean and ~1 variance
ln = LayerNorm(emb_dim=5)
out = ln(torch.randn(2, 5))     # arbitrary activations, just to verify normalization
print("mean:", out.mean(dim=-1).detach())
print("var: ", out.var(dim=-1, unbiased=False).detach())

mean: tensor([-7.1526e-08,  7.4506e-10])
var:  tensor([0.9999, 1.0000])


## 4.3 A feed-forward network with GELU activation

After attention mixes information *across* tokens, the feed-forward network
processes each token *independently*, expanding to 4x the width and back. It uses
**GELU**, a smooth alternative to ReLU used in GPT-2.

In [5]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),  # expand 768 -> 3072
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),  # project 3072 -> 768
        )

    def forward(self, x):
        return self.layers(x)


ffn = FeedForward(GPT_CONFIG_124M)
# a batch of 2 sequences x 3 tokens, each a 768-dim embedding
print("feed-forward output shape:", ffn(torch.rand(2, 3, 768)).shape)

feed-forward output shape: torch.Size([2, 3, 768])


## 4.4 Adding shortcut connections

A **shortcut** (residual) connection adds a layer's input to its output:
`x = x + layer(x)`. This gives gradients a "highway" straight back through the
network, which is what makes very deep models trainable. The demo below shows
gradients are far healthier *with* shortcuts than without.

In [6]:
class ExampleDeepNN(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[i], layer_sizes[i+1]), GELU())
            for i in range(len(layer_sizes) - 1)
        ])

    def forward(self, x):
        for layer in self.layers:
            out = layer(x)
            x = x + out if (self.use_shortcut and x.shape == out.shape) else out
        return x


def print_gradients(model, x):
    output = model(x)
    loss = torch.nn.functional.mse_loss(output, torch.tensor([[0.]]))
    loss.backward()
    for name, param in model.named_parameters():
        if "weight" in name:
            print(f"{name}: grad magnitude {param.grad.abs().mean().item():.6f}")


sizes = [3, 3, 3, 3, 3, 1]
sample = torch.tensor([[1., 0., -1.]])

print("WITHOUT shortcuts (gradients vanish in early layers):")
torch.manual_seed(123)
print_gradients(ExampleDeepNN(sizes, use_shortcut=False), sample)
print("\nWITH shortcuts (gradients stay healthy):")
torch.manual_seed(123)
print_gradients(ExampleDeepNN(sizes, use_shortcut=True), sample)

WITHOUT shortcuts (gradients vanish in early layers):


layers.0.0.weight: grad magnitude 0.000202
layers.1.0.weight: grad magnitude 0.000120
layers.2.0.weight: grad magnitude 0.000715
layers.3.0.weight: grad magnitude 0.001399
layers.4.0.weight: grad magnitude 0.005050

WITH shortcuts (gradients stay healthy):
layers.0.0.weight: grad magnitude 0.221698
layers.1.0.weight: grad magnitude 0.206941
layers.2.0.weight: grad magnitude 0.328970
layers.3.0.weight: grad magnitude 0.266573
layers.4.0.weight: grad magnitude 1.325854


## 4.5 The transformer block

The transformer block wires everything together. Note the **Pre-LayerNorm**
design (norm *before* each sub-layer) and the two shortcut connections:

```
x -> norm1 -> attention -> dropout -> (+x)
  -> norm2 -> feed-forward -> dropout -> (+x)
```

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"], num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x                       # --- attention sub-layer ---
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x                       # --- feed-forward sub-layer ---
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


# A real input: tokenize two short sentences, then embed them (the actual pipeline).
batch = torch.stack([
    torch.tensor(tokenizer.encode("Every effort moves you")),   # 4 tokens
    torch.tensor(tokenizer.encode("Every day holds a")),        # 4 tokens
])   # (2, 4) token IDs

torch.manual_seed(123)
demo_token_emb = torch.nn.Embedding(GPT_CONFIG_124M["vocab_size"], GPT_CONFIG_124M["emb_dim"])
x = demo_token_emb(batch)                      # (2, 4, 768) real token embeddings

block = TransformerBlock(GPT_CONFIG_124M)
print("token IDs:\n", batch.tolist())
print("block in/out shape:", x.shape, "->", block(x).shape)  # shape preserved

token IDs:
 [[6109, 3626, 6100, 345], [6109, 1110, 6622, 257]]
block in/out shape: torch.Size([2, 4, 768]) -> torch.Size([2, 4, 768])


## 4.6 Coding the GPT model

Finally, the whole model:

1. **Token embeddings** + **positional embeddings** (Chapter 2).
2. A stack of `n_layers` **transformer blocks**.
3. A final **layer norm**.
4. An **output head** (linear) projecting to vocabulary-sized logits.

In [8]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

# Real forward pass: the tokenized 'batch' from above -> logits over the vocabulary.
logits = model(batch)
print("Input (token IDs) shape:", tuple(batch.shape))    # (2, 4)
print("Output logits shape:    ", tuple(logits.shape))   # (2, 4, 50257)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"Approx model size: {total_params * 4 / 1024**2:.1f} MB (float32)")

Input (token IDs) shape: (2, 4)
Output logits shape:     (2, 4, 50257)

Total parameters: 163,009,536
Approx model size: 621.8 MB (float32)


> **Why ~163M and not 124M?** GPT-2 *ties* the token-embedding weights with the
> output head (they share one matrix). Counting them once gives the famous 124M.
> We keep them separate here for clarity; Chapter 5 loads the tied OpenAI weights.

## 4.7 Generating text

The model outputs **logits**: a score for every vocabulary token at every
position. To generate, we repeatedly: take the logits at the last position, pick
the most likely next token (argmax), append it, and feed the longer sequence back
in.

In [9]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]          # crop to context window
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]                   # only the last position
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # greedy pick
        idx = torch.cat((idx, idx_next), dim=1)     # append and repeat
    return idx


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(encoded).unsqueeze(0)       # add batch dim

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

In [10]:
tokenizer = tiktoken.get_encoding("gpt2")
start_context = "Every effort moves you"

model.eval()
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"],
)
print("Output:", token_ids_to_text(token_ids, tokenizer))

Output: Every effort moves you Aeiman Byeswickattributeometer inspector Normandy freezerigrate


The output is nonsense — as expected, since the model has **random** weights. The
architecture is complete and correct; it just hasn't learned anything yet.

## Summary

You now have a full, working GPT-2 124M architecture: embeddings → 12 transformer
blocks (each: layer norm → multi-head attention → shortcut → layer norm →
feed-forward → shortcut) → final norm → output head, plus a text-generation loop.

### Try it yourself
- Make a tiny model (`emb_dim=32, n_heads=2, n_layers=2`) and re-count parameters.
- Increase `max_new_tokens` and confirm generation still runs (just longer gibberish).

**Next:** `04_ch5_pretraining.ipynb` — actually **train** the model so its output
becomes real language, and load OpenAI's pretrained GPT-2 weights.